# PySpark 6: joins

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-06-joins.ipynb)

Revenue by product category needs order lines joined to products, orders and customers. A wrong join type
silently drops revenue; a duplicate key silently doubles it; a join on a skewed key makes the job run for hours.
Joins are where most pipeline bugs and most slow stages come from.

This lesson chooses join types by what should happen to unmatched rows, handles null keys and duplicate column
names, checks join cardinality, joins on ranges and points in time, and looks at how Spark executes a join:
broadcast, sort-merge and shuffled hash, and the hints that pick between them.

Each section explains how something works, where it matters and what to watch out for, then runs small examples. Read each output, then the **What to notice** note under it. Run the cells in order.

**Prerequisites:** [PySpark 2: DataFrame essentials](pyspark-02-dataframe-essentials.ipynb) and [PySpark 3: types, functions and nulls](pyspark-03-types-functions-and-nulls.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `customers` (one row per customer; customer 4 has no region, and 7 and 8 have never ordered), `orders` (one row per order; order 12 lands just after midnight UTC), `order_items` (one row per order line; each order's lines add up to its amount) and `products` (one row per SKU; `SKU-4014` is missing, so one order line has no product): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#sample-data-used-in-every-module).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),  # a bulk reorder
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing: an order line with no product row
], "sku string, category string, list_price decimal(8,2)")

In [ ]:
import contextlib
import io


def plan_of(df, mode="formatted"):
    """Return the text `df.explain(mode)` prints, so a cell can pick out the lines that matter."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue()

## Choose a join type by what happens to unmatched rows

`left.join(right, on, how)` combines rows whose keys match. The join type decides what happens to rows that
find no match:

| `how` | keeps | typical use |
| --- | --- | --- |
| `inner` (default) | matched rows only | facts that must have a dimension row |
| `left` | every left row; right columns null when unmatched | enrich facts without losing any |
| `right` | every right row | rare; swap the inputs and use `left` |
| `full` | every row from both sides | reconcile a source against a target |
| `left_semi` | left rows that have a match; left columns only; never duplicated | "customers who ordered" |
| `left_anti` | left rows with no match | "customers who never ordered", missing keys |
| `cross` | every combination | small calendars, test grids |

Before writing a join, state the **grain** of each input (what one row is) and of the output.

**Where it matters**
- **Enrichment:** order lines left-joined to products keep lines whose product is missing.
- **Data quality:** `left_anti` lists fact keys missing from a dimension.
- **Reconciliation:** a `full` join classifies every key as matched, missing or extra.

**Watch out for**
- An `inner` join drops unmatched facts without any error, and with them their revenue.
- `cross` joins of real tables explode: a million rows times a million is a trillion.

### Example

Join `customers` to `orders` with each type and compare the row counts.

In [ ]:
for how in ["inner", "left", "right", "full", "left_semi", "left_anti", "cross"]:
    joined = customers.crossJoin(orders) if how == "cross" else customers.join(orders, "customer_id", how)
    print(f"{how:<10} {joined.count():>3} rows")

**What to notice:** `inner` and `right` return the 12 orders. `left` and `full` add customers 7 and 8 with null order columns: 14.
`left_semi` returns the 6 customers with orders, each once, even though customer 1 has three orders.
`left_anti` returns the 2 who never ordered. `cross` pairs every customer with every order: 8 x 12 = 96.

### Example: enrich without losing rows

Enrich order lines with their product. One line's SKU is missing from `products`.

In [ ]:
lines = order_items.withColumn("line_amount", F.col("qty") * F.col("unit_price"))
inner = lines.join(products, "sku", "inner")
left = lines.join(products, "sku", "left")

print("order lines:", lines.count(), "| inner:", inner.count(), "| left:", left.count())
print("revenue - lines:", lines.agg(F.sum("line_amount")).first()[0],
      "| inner:", inner.agg(F.sum("line_amount")).first()[0],
      "| left:", left.agg(F.sum("line_amount")).first()[0])
left.where("category IS NULL").select("order_id", "sku", "line_amount").show()

**What to notice:** The inner join lost the line for `SKU-4014` and with it 41.40 of revenue, with no error. The left join kept it
with a null category, which a report can show as "unknown" and a quality check can flag.

## Handle null keys and duplicate column names

Join conditions use SQL equality, and `NULL = NULL` is not true, so **null keys never match**. When nulls should
match each other, use the null-safe operator: `left["k"].eqNullSafe(right["k"])`.

How you write the condition also decides the output columns:

- `on="k"` or `on=["k1", "k2"]` keeps **one** copy of each key column;
- an expression such as `left["k"] == right["k"]` keeps **both**, and a later reference to `k` is ambiguous.
  Use aliases (`df.alias("o")`) and select what you need.

**Where it matters**
- **Composite keys with optional parts,** such as `(country, region)` where region can be null.
- **Self joins** (employees to their managers), which always need aliases.

**Watch out for**
- Many rows with a null or placeholder key (`-1`, `UNKNOWN`) all hash to one partition and skew the join.
- `AMBIGUOUS_REFERENCE` after a join means two columns share a name; qualify them through aliases.

### Example

Join on a nullable key with `==` and with `eqNullSafe`, then compare the columns produced by joining on a name and on an expression.

In [ ]:
left = spark.createDataFrame([(1, "a"), (None, "b")], "k int, left_val string")
right = spark.createDataFrame([(1, "x"), (None, "y")], "k int, right_val string")

print("== join:")
left.join(right, "k").show()
print("null-safe join:")
left.join(right, left["k"].eqNullSafe(right["k"])).show()

print("on a name:      ", left.join(right, "k").columns)
print("on an expression:", left.join(right, left["k"] == right["k"]).columns)
l, r = left.alias("l"), right.alias("r")
print("aliased:        ", l.join(r, F.col("l.k") == F.col("r.k")).select("l.k", "left_val", "right_val").columns)

**What to notice:** The plain join returned one row: the two null keys did not meet. `eqNullSafe` returned both. Joining on the name
`k` produced a single `k` column; joining on an expression produced two, and selecting through the `l` and `r`
aliases picks the one you mean.

## Check join cardinality

**Cardinality** is how many rows on one side match a row on the other: one-to-one, many-to-one, one-to-many or
many-to-many. A fact table joined to a dimension should be **many-to-one**; if the dimension has two rows for
a key, every matching fact is duplicated and any sum after the join is inflated.

Check before joining (is the key unique?) and after (did the row count change?). Count unmatched keys
separately: in an inner join, a duplicate and a missing match can cancel out in a total row count.

**Where it matters**
- **History tables** with several versions per key: join to the version valid at the event time (below).
- **Vendor dimensions** that arrive with accidental duplicates.

**Watch out for**
- Fix duplicates with a business rule (latest version, valid version), not with an arbitrary `dropDuplicates`.
- Many-to-many joins are sometimes right (orders to promotions), but then aggregate before or after deliberately.

### Example

A customer dimension accidentally has two rows for customer 3. Check uniqueness, then see what the join does to revenue.

In [ ]:
customers_dup = customers.unionByName(customers.where("customer_id = 3").withColumn("region", F.lit("Central")))

customers_dup.groupBy("customer_id").count().where("count > 1").show()

joined = orders.join(customers_dup, "customer_id", "left")
print("orders:", orders.count(), "| after join:", joined.count())
print("revenue before:", orders.agg(F.sum("amount")).first()[0], "| after:", joined.agg(F.sum("amount")).first()[0])

**What to notice:** The uniqueness check flags customer 3 before anything is joined. Joining anyway turned customer 3's two orders
into four rows, and total revenue rose by exactly their 459.00. Nothing failed; the numbers were simply wrong.

## Join on ranges and points in time

A join condition can be any boolean expression, not only equality:

- **range joins:** an amount falls into a price band (`amount >= low AND amount < high`);
- **as-of joins:** an event takes the version of a dimension that was valid at the event time
  (`event_ts >= valid_from AND event_ts < valid_to`).

Always include an equality on a key when there is one (`customer_id` plus the time range). A join with only
range conditions cannot be hashed, so Spark compares every pair of rows (a nested loop).

**Where it matters**
- **Slowly changing dimensions:** the customer's segment at the time of each order.
- **Banding:** tax rates, price tiers, time-of-day buckets.

**Watch out for**
- Use half-open intervals `[from, to)` so a value exactly on a boundary matches one band, not two.
- Overlapping versions in the dimension produce duplicate matches; check for overlaps first.

### Example

Put each order into a size band, then attach the customer segment that was valid when each order was placed.

In [ ]:
bands = spark.createDataFrame(
    [("small", 0, 100), ("mid", 100, 1000), ("large", 1000, 1_000_000)], "band string, low int, high int"
)
banded = orders.join(F.broadcast(bands), (orders.amount >= bands.low) & (orders.amount < bands.high), "left")
banded.groupBy("band").count().orderBy("band").show()

segments = spark.createDataFrame(
    [(1, "standard", datetime(2024, 1, 1), datetime(2026, 1, 5, 10, 0)),
     (1, "premium", datetime(2026, 1, 5, 10, 0), None)],
    "customer_id int, segment string, valid_from timestamp, valid_to timestamp",
)
o, s = orders.alias("o"), segments.alias("s")
as_of = o.join(
    s,
    (F.col("o.customer_id") == F.col("s.customer_id"))
    & (F.col("o.order_ts") >= F.col("s.valid_from"))
    & (F.col("o.order_ts") < F.coalesce(F.col("s.valid_to"), F.lit("9999-12-31").cast("timestamp"))),
    "left",
)
as_of.where("o.customer_id = 1").select("o.order_id", "o.order_ts", "s.segment").orderBy("order_id").show()

**What to notice:** Every order landed in exactly one band, because the intervals are half-open. Customer 1 became premium at
10:00 on 5 January: order 1 (09:15) took the standard version and orders 2 and 3 the premium one. A null
`valid_to` means "still current" and is replaced with a far-future date in the condition.

## How Spark executes a join

Spark picks a physical strategy for each join:

- **Broadcast hash join:** the small side is copied to every executor and the large side is joined in place,
  with **no shuffle**. Chosen automatically when a side is estimated below
  `spark.sql.autoBroadcastJoinThreshold` (10 MB), or when you hint `F.broadcast(df)`.
- **Sort-merge join:** both sides are shuffled by the key and sorted, then merged. The default for two large
  sides; it scales and can spill to disk.
- **Shuffled hash join:** both sides are shuffled, and the smaller side of each partition is built into a hash
  table. Avoids the sort; chosen by adaptive execution or a `shuffle_hash` hint.
- **Broadcast nested loop join:** for conditions without an equality; compares every pair. Avoid on large data.

**Where it matters**
- **Fact-to-dimension joins:** broadcasting a 20 MB dimension avoids shuffling a 3 TB fact table.
- **Two large tables:** sort-merge, with the keys chosen so each side is shuffled once.

**Watch out for**
- Broadcasting something that is not small runs the driver and executors out of memory. Check its size.
- A broadcast does not fix skew on the large side's key; see [lesson 13](pyspark-13-performance-tuning.ipynb).
- Confirm the strategy in the plan, not in your code.

### Example

Compare the plans for the same join with broadcasting disabled, with a broadcast hint, and with a merge hint.

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # disable automatic broadcasts for the comparison

def strategy(df):
    plan = plan_of(df, "simple")
    names = ["BroadcastHashJoin", "SortMergeJoin", "ShuffledHashJoin", "BroadcastNestedLoopJoin", "CartesianProduct"]
    return next((name for name in names if name in plan), "other")

plain = orders.join(customers, "customer_id")
hinted = orders.join(F.broadcast(customers), "customer_id")
shuffle_hash = orders.join(customers.hint("shuffle_hash"), "customer_id")
ranged = orders.join(customers, orders.amount > customers.customer_id * 100)

for name, df in [("no hint", plain), ("broadcast()", hinted), ("shuffle_hash hint", shuffle_hash), ("range only", ranged)]:
    print(f"{name:<18} {strategy(df)}")
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
print(f"{'range + broadcast()':<18} {strategy(orders.join(F.broadcast(customers), orders.amount > customers.customer_id * 100))}")
hinted.explain()

**What to notice:** With automatic broadcasts disabled, the plain join became a `SortMergeJoin`, which shuffles and sorts both sides.
The `broadcast()` hint produced a `BroadcastHashJoin`: the plan shows a `BroadcastExchange` for `customers` and
no exchange at all on the `orders` side. The `shuffle_hash` hint chose a `ShuffledHashJoin`. A condition with no
equality cannot be hashed or sorted: without a broadcast it became a `CartesianProduct` (every pair of rows),
and with a broadcast hint a `BroadcastNestedLoopJoin`, which still compares every pair but against a copy of the
small side on each executor.

## Join several tables

Real queries chain joins. Keep each one deliberate: join facts to dimensions with `left` joins on unique keys,
filter and select columns **before** joining so less data moves, and aggregate to the grain you need before
joining two facts together. Spark's optimizer reorders some joins, but it cannot fix a wrong join type or a
fan-out.

**Where it matters**
- **Star schemas:** one fact table and several dimensions.
- **Reports** that combine order lines, products, orders and customers.

**Watch out for**
- Joining two facts (order lines and payments) on a shared key multiplies rows; aggregate one side first.
- Repeated column names (`amount` in two tables) need renaming before the join.

### Example

Compute revenue by region and product category from four tables, keeping lines whose product or region is missing.

In [ ]:
report = (
    order_items
    .withColumn("line_amount", F.col("qty") * F.col("unit_price"))
    .join(products.select("sku", "category"), "sku", "left")
    .join(orders.select("order_id", "customer_id", "status"), "order_id", "left")
    .join(customers.select("customer_id", "region"), "customer_id", "left")
    .where(F.col("status").isin("paid", "shipped"))
    .groupBy(F.coalesce("region", F.lit("unknown")).alias("region"),
             F.coalesce("category", F.lit("unknown")).alias("category"))
    .agg(F.sum("line_amount").alias("revenue"))
    .orderBy("region", "category")
)
report.show()
print("report total:", report.agg(F.sum("revenue")).first()[0])

**What to notice:** The report total, 2,273.90, equals the revenue of paid and shipped orders: no line was lost or duplicated,
because every join was a `left` join on a unique key. Customer 4's missing region shows as `unknown`, and order
12's line for the missing `SKU-4014` shows as an `unknown` category instead of disappearing.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** Which join type returns customers who never ordered?

**A.** `left_anti`: left rows with no match on the right.

**Q2.** How is `left_semi` different from an `inner` join?

**A.** It returns only left columns and never duplicates a left row, however many matches the right side has.

**Q3.** Why do rows with null keys disappear from a join?

**A.** `NULL = NULL` is not true, so null keys never match. Use `eqNullSafe` when they should.

**Q4.** Revenue doubled after a join. What do you check first?

**A.** Whether the dimension key is unique: `groupBy(key).count().where('count > 1')`.

**Q5.** When does Spark broadcast a join input, and what is the risk?

**A.** When its estimated size is under `spark.sql.autoBroadcastJoinThreshold` (10 MB) or it is hinted. If it is not really small, the driver and executors run out of memory.

**Q6.** Why add an equality on the key to an as-of join?

**A.** Without an equality Spark cannot hash or sort the join, so it compares every pair of rows.

## Key takeaways

- State each input's grain, then pick the join type by what should happen to unmatched rows.
- Left joins on unique keys keep facts and their revenue; check uniqueness and row counts.
- Null keys never match with `==`; use `eqNullSafe` when they should.
- Use half-open intervals for range and as-of joins, and include the key equality.
- Broadcast only small sides, and confirm the strategy in the plan.

**Next:** Practise this module in [PySpark drills 6: joins](pyspark-drills-06-joins.ipynb), then continue with [PySpark 7: aggregations and windows](pyspark-07-aggregations-and-windows.ipynb).